# Giai đoạn 1 — Hồ sơ đối chiếu cho nhóm trưởng
**Phan Đồng Minh Quân — MSSV 3123580038**  
**Phạm vi:** tìm bộ dữ liệu gian lận, phân loại thang đo Stevens, làm sạch và xử lý missing trên **training data**. Không huấn luyện mô hình trong giai đoạn này.

## Cách review và chạy lại
- Notebook này chứa cả giải thích, code và **output đã chạy thực tế**. Đọc lần lượt Bước 1–5; Bước 5 tổng hợp checklist.
- Giải nén **toàn bộ ZIP** rồi mở notebook tại thư mục gốc của gói. Hai CSV ở `data/raw/`, Parquet đã chạy ở `data/processed/`.
- Trong môi trường Python đang dùng để mở notebook, cài `python -m pip install -r requirements.txt`; chọn kernel đúng môi trường, **Restart Kernel → Run All**.
- Mặc định mỗi lô 1.000 dòng. Identity nhỏ hơn được giữ trong RAM; transaction/Parquet được đọc theo lô. Không bảo đảm chạy trên mọi cấu hình; số đo RAM của lần chạy kiểm chứng nằm trong `review_results/execution_environment.json`.
- Chạy lại tạo lại Parquet và bảng CSV; không sửa CSV nguồn. Nếu thấy trùng khóa, chương trình xuất báo cáo rồi dừng để xem xét.
- Không cần các bản ZIP/notebook cũ, file Python riêng, Markdown hay PNG ngoài gói này.

## Bộ dữ liệu và nguồn tham khảo
IEEE-CIS Fraud Detection do IEEE-CIS/Vesta cung cấp qua Kaggle. Dùng `train_transaction.csv` và `train_identity.csv`; `sample_submission.csv` chỉ là mẫu nộp dự đoán.

1. [Kaggle — Data](https://www.kaggle.com/c/ieee-fraud-detection/data): khóa ghép, target, danh sách categorical và thời gian tương đối.
2. [Mô tả của ban tổ chức](https://www.kaggle.com/competitions/ieee-fraud-detection/discussion/101203): ý nghĩa nhóm biến; nhiều biến bị ẩn nghĩa. Trang có thể yêu cầu đăng nhập; không xem mô tả nhóm là codebook cho từng cột.
3. Stevens, S. S. (1946), *On the Theory of Scales of Measurement*, Science, 103(2684), 677–680. [DOI](https://doi.org/10.1126/science.103.2684.677).

**Tiêu chí Stevens:** danh nghĩa = nhóm; thứ bậc = có thứ tự; khoảng = chênh lệch có nghĩa, gốc tùy ý; tỷ lệ = có gốc 0 có ý nghĩa. Kiểu `int`/`float` không quyết định thang đo. Không ép bộ dữ liệu phải có đủ cả bốn thang.


## 1. Chuẩn bị và xác định chính sách đọc
Ô code này nạp thư viện, kiểm tra hai file nguồn và tạo thư mục kết quả. Chỉ ô rỗng được hiểu là missing khi đọc CSV; chuỗi trắng ở categorical được chuẩn hóa ở Bước 3. Giữ `float64` cho biến số, đọc mã phân loại thành chuỗi để không diễn giải mã là số đo.

SQLite là kho kiểm tra tạm trên đĩa, giúp so khóa toàn bộ dữ liệu mà không nạp bảng 437 cột vào RAM. `CHUNK_SIZE` chỉ điều khiển số dòng mỗi lô; không lấy mẫu hoặc bỏ dòng.


In [1]:
"""Các hàm được chép vào notebook nghiệm thu; notebook không cần import file này."""
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.metadata
import json
import platform
import re
import sqlite3
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

UNKNOWN = "Không xác định"
NO_IDENTITY = "Không có bản ghi identity"
MISSING_IDENTITY_FIELD = "Thiếu trong bản ghi identity"
MISSING_LABELS = [UNKNOWN, NO_IDENTITY, MISSING_IDENTITY_FIELD]
KAGGLE_URL = "https://www.kaggle.com/c/ieee-fraud-detection/data"
DETAIL_URL = "https://www.kaggle.com/competitions/ieee-fraud-detection/discussion/101203"



def categorical(name):
    return name in {"ProductCD", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo", "IdentityStatus"} or bool(re.fullmatch(r"card[1-6]|addr[12]|M[1-9]|id_(?:1[2-9]|2[0-9]|3[0-8])", name))

def read_policy(columns):
    # Không giảm float64 xuống float32: tránh làm tròn số tiền/đặc trưng nguồn.
    return {c: "string" if categorical(c) else "int64" if c in {"TransactionID", "isFraud", "TransactionDT"} else "float64" for c in columns}

def csv_options(columns):
    # Quy ước missing công khai: ô rỗng; khoảng trắng ở categorical được chuẩn hóa sau.
    return dict(dtype=read_policy(columns), keep_default_na=False, na_values=[""], float_precision="round_trip")

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def save_csv(frame, path):
    frame.to_csv(path, index=False, encoding="utf-8-sig")

def open_audit_db(path):
    db = sqlite3.connect(path)
    db.execute("PRAGMA cache_size=-8192")  # Giới hạn cache SQLite khoảng 8 MiB.
    db.execute("PRAGMA temp_store=FILE")
    return db

from IPython.display import display
ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "review_results"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 1000
TX_PATH = RAW / "train_transaction.csv"
ID_PATH = RAW / "train_identity.csv"
PARQUET_PATH = PROCESSED / "fraud_train_clean.parquet"
for path in (TX_PATH, ID_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Hãy giải nén toàn bộ gói trước khi Run All.")
STARTED_AT = datetime.now(timezone.utc).isoformat()
START_TIMER = time.perf_counter()
DB = open_audit_db(REPORTS / "audit_work.sqlite")
print("Bắt đầu UTC:", STARTED_AT)
print("Python:", platform.python_version(), "| Pandas:", pd.__version__, "| PyArrow:", pa.__version__)
print("Số dòng mỗi lô:", CHUNK_SIZE)
display(pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size} for p in (TX_PATH, ID_PATH)]))


Bắt đầu UTC: 2026-10-14T12:57:57.255980+00:00
Python: 3.14.6 | Pandas: 3.0.3 | PyArrow: 23.0.1
Số dòng mỗi lô: 1000


,file,bytes
0,train_transaction.csv,683351067
1,train_identity.csv,26529680


## 2. Kiểm tra nguồn độc lập và báo cáo trùng
Đọc hai CSV bằng `csv.reader`, độc lập với Pandas ở bước làm sạch. Đếm số dòng, số ID duy nhất, số fraud; lưu khóa và nhãn vào SQLite. Đếm missing gốc và chuỗi trắng theo cột, đồng thời ghi SHA256 của file.

**Trùng được tách rõ:** số nhóm ID trùng, số dòng thuộc nhóm trùng, số dòng dư sau lần xuất hiện đầu; ba chỉ số tương tự cho trùng toàn dòng. Toàn dòng bao gồm `TransactionID` và mọi trường CSV gốc (so giá trị chuỗi sau khi giải mã CSV, chưa trim/chuyển kiểu). Hash chỉ lọc ứng viên; ứng viên trùng được so lại đủ trường. Hai giao dịch có cùng đặc trưng nhưng khác ID không bị xem là trùng toàn dòng.

Lưu bảng trùng trước khi kiểm tra điều kiện ghép; nếu khóa thiếu/trùng/không hợp lệ thì **dừng, không tự xóa**. Độ phủ identity được tính từ giao khóa nguồn. Missing do không có identity và missing trong record có thật được tính riêng.


In [2]:
def full_row_hash(row):
    # Hash chỉ tìm ứng viên. Mọi nhóm hash trùng sẽ được so lại toàn bộ trường.
    return hashlib.sha256(json.dumps(row, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).digest()

def duplicate_stats(db, table, path=None):
    # Các table đều do chương trình tạo; không nhận tên table từ người dùng.
    groups, involved, excess = db.execute(f"SELECT COUNT(*), COALESCE(SUM(n),0), COALESCE(SUM(n-1),0) FROM (SELECT COUNT(*) n FROM {table} WHERE id IS NOT NULL GROUP BY id HAVING COUNT(*)>1)").fetchone()
    full_groups = full_involved = full_excess = 0
    if path is not None:
        candidates = {r[0] for r in db.execute(f"SELECT row_hash FROM {table} GROUP BY row_hash HAVING COUNT(*)>1")}
        exact = Counter()
        if candidates:
            with Path(path).open(encoding="utf-8-sig", newline="") as f:
                reader = csv.reader(f)
                next(reader)
                for row in reader:
                    if full_row_hash(row) in candidates:
                        exact[tuple(row)] += 1
            counts = [n for n in exact.values() if n > 1]
            full_groups, full_involved, full_excess = len(counts), sum(counts), sum(n - 1 for n in counts)
        method = "SHA256 các trường CSV + đối chiếu toàn dòng cho nhóm hash trùng; gồm TransactionID"
    else:
        # Toàn dòng trùng => ID trùng. Nếu có ID trùng, đọc lại các dòng đó và so đủ cột.
        candidate_ids = {r[0] for r in db.execute(f"SELECT id FROM {table} GROUP BY id HAVING COUNT(*)>1")}
        method = "Kiểm tra ID toàn cục; nếu có ID trùng, đối chiếu đủ cột Parquet của các dòng ứng viên"
        if candidate_ids:
            exact = Counter()
            for batch in pq.ParquetFile(PARQUET_PATH).iter_batches(batch_size=1000):
                frame = batch.to_pandas()
                frame = frame[frame.TransactionID.isin(candidate_ids) | frame.TransactionID.isna()]
                for row in frame.itertuples(index=False, name=None):
                    key = tuple(None if pd.isna(v) else v for v in row)
                    exact[key] += 1
            counts = [n for n in exact.values() if n > 1]
            full_groups, full_involved, full_excess = len(counts), sum(counts), sum(n - 1 for n in counts)
    return dict(dataset=table, duplicate_id_groups=groups, duplicate_id_rows_involved=involved,
                duplicate_id_extra_rows=excess, duplicate_full_row_groups=full_groups,
                duplicate_full_rows_involved=full_involved, duplicate_full_extra_rows=full_excess,
                full_row_definition=method, action="Giữ nguyên; nếu khóa trùng thì dừng để xem xét, không tự xóa")

def audit_source(path, table, db, batch_size=5000):
    """Lượt đọc CSV độc lập bằng thư viện csv; không dùng bộ đếm của pipeline."""
    db.execute(f"DROP TABLE IF EXISTS {table}")
    db.execute(f"CREATE TABLE {table}(id INTEGER, fraud INTEGER, amount REAL, dt INTEGER, row_hash BLOB)")
    empty, normalized, collision = Counter(), Counter(), Counter()
    invalid_id = missing_id = invalid_label = 0
    rows = 0
    pending = []
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        columns = next(reader)
        key = columns.index("TransactionID")
        label = columns.index("isFraud") if "isFraud" in columns else None
        amount = columns.index("TransactionAmt") if "TransactionAmt" in columns else None
        dt = columns.index("TransactionDT") if "TransactionDT" in columns else None
        cat_indices = [i for i, c in enumerate(columns) if categorical(c)]
        for row in reader:
            if len(row) != len(columns):
                raise ValueError(f"{path}: dòng {rows + 2} không đủ/sai số trường")
            rows += 1
            for i, value in enumerate(row):
                if value == "":
                    empty[columns[i]] += 1
                    normalized[columns[i]] += 1
            for i in cat_indices:
                if row[i] != "" and row[i].strip() == "":
                    normalized[columns[i]] += 1
                if row[i].strip() in MISSING_LABELS:
                    collision[columns[i]] += 1
            try:
                transaction_id = int(row[key])
            except ValueError:
                transaction_id = None
                missing_id += int(row[key].strip() == "")
                invalid_id += int(row[key].strip() != "")
            fraud = None
            if label is not None:
                if row[label] in {"0", "1"}:
                    fraud = int(row[label])
                else:
                    invalid_label += 1
            pending.append((transaction_id, fraud, float(row[amount]) if amount is not None and row[amount] else None,
                            int(row[dt]) if dt is not None and row[dt] else None, full_row_hash(row)))
            if len(pending) >= batch_size:
                db.executemany(f"INSERT INTO {table} VALUES(?,?,?,?,?)", pending)
                pending.clear()
            if rows % 100000 == 0:
                print(f"Kiểm tra nguồn {table}: {rows:,} dòng", flush=True)
    if pending:
        db.executemany(f"INSERT INTO {table} VALUES(?,?,?,?,?)", pending)
    db.execute(f"CREATE INDEX {table}_id_idx ON {table}(id)")
    db.execute(f"CREATE INDEX {table}_hash_idx ON {table}(row_hash)")
    db.commit()
    observed_rows, distinct_ids, frauds = db.execute(f"SELECT COUNT(*), COUNT(DISTINCT id), SUM(fraud=1) FROM {table}").fetchone()
    return dict(path=str(path), columns=columns, rows=observed_rows, distinct_ids=distinct_ids,
                frauds=frauds, missing_id=missing_id, invalid_id=invalid_id, invalid_label=invalid_label,
                empty=dict(empty), normalized_missing=dict(normalized), reserved_label_collisions=sum(collision.values()),
                sha256=file_hash(path), duplicates=duplicate_stats(db, table, path))

def validate_sources(transaction, identity):
    issues = []
    for name, audit in [("transaction", transaction), ("identity", identity)]:
        for key in ["missing_id", "invalid_id", "invalid_label", "reserved_label_collisions"]:
            if audit[key]:
                issues.append(f"{name}: {key}={audit[key]}")
        if audit["duplicates"]["duplicate_id_extra_rows"]:
            issues.append(f"{name}: TransactionID trùng")
    if issues:
        raise ValueError("Đã lưu báo cáo; không xóa dòng. Cần xem xét: " + "; ".join(issues))

def source_join_expectations(db, tx_audit, id_audit, identity_path):
    # Tính độ phủ bằng phép giao khóa từ hai file nguồn, trước khi tạo Parquet.
    matched = {r[0] for r in db.execute("SELECT i.id FROM source_identity i INNER JOIN source_transaction t ON t.id=i.id")}
    no_identity = tx_audit["rows"] - len(matched)
    within_empty, within_normalized = Counter(), Counter()
    with Path(identity_path).open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if int(row["TransactionID"]) in matched:
                for c, value in row.items():
                    if c != "TransactionID":
                        within_empty[c] += int(value == "")
                        within_normalized[c] += int(value == "" or (categorical(c) and value.strip() == ""))
    before = Counter(tx_audit["empty"])
    unknown = Counter(tx_audit["normalized_missing"])
    for c in id_audit["columns"]:
        if c != "TransactionID":
            before[c] = within_empty[c] + no_identity
            unknown[c] = within_normalized[c] + no_identity
    return dict(matched=len(matched), absent=no_identity, orphan_identity=id_audit["rows"]-len(matched),
                before=dict(before), unknown=dict(unknown), within_empty=dict(within_empty), within_normalized=dict(within_normalized))

tx_audit = audit_source(TX_PATH, "source_transaction", DB)
id_audit = audit_source(ID_PATH, "source_identity", DB)
source_duplicates = pd.DataFrame([tx_audit["duplicates"], id_audit["duplicates"]])
save_csv(source_duplicates, REPORTS / "duplicates.csv")
display(source_duplicates.drop(columns=["full_row_definition", "action"]))
validate_sources(tx_audit, id_audit)
join_expected = source_join_expectations(DB, tx_audit, id_audit, ID_PATH)
original_columns = tx_audit["columns"] + [c for c in id_audit["columns"] if c != "TransactionID"]
print("Giao dịch nguồn:", tx_audit["rows"], "| ID duy nhất:", tx_audit["distinct_ids"], "| Fraud:", tx_audit["frauds"])
print("Có identity:", join_expected["matched"], "| Không có:", join_expected["absent"], "| Identity không ghép được:", join_expected["orphan_identity"])


Kiểm tra nguồn source_transaction: 100,000 dòng


Kiểm tra nguồn source_transaction: 200,000 dòng


Kiểm tra nguồn source_transaction: 300,000 dòng


Kiểm tra nguồn source_transaction: 400,000 dòng


Kiểm tra nguồn source_transaction: 500,000 dòng


Kiểm tra nguồn source_identity: 100,000 dòng


,dataset,duplicate_id_groups,duplicate_id_rows_involved,duplicate_id_extra_rows,duplicate_full_row_groups,duplicate_full_rows_involved,duplicate_full_extra_rows
0,source_transaction,0,0,0,0,0,0
1,source_identity,0,0,0,0,0,0


Giao dịch nguồn: 590540 | ID duy nhất: 590540 | Fraud: 20663
Có identity: 144233 | Không có: 446307 | Identity không ghép được: 0


## 3. Làm sạch theo lô và ghi Parquet
Ghép trái transaction với identity theo `TransactionID`, xác thực quan hệ 1–1. Không bỏ cột, không bỏ giao dịch, không loại outlier, không cân bằng nhãn, không điền trung bình/median vào dữ liệu số.

| Loại ô thiếu | Cách xử lý |
|---|---|
| Biến số | Giữ null/NaN; người phân tích phải báo số quan sát hữu hiệu |
| Categorical thuộc transaction | Trim chuỗi; ô rỗng → “Không xác định” |
| Categorical identity nhưng không có record | “Không có bản ghi identity” |
| Có record identity nhưng trường categorical bị thiếu | “Thiếu trong bản ghi identity” |

Ba cột dẫn xuất: `HasIdentity`, `IdentityStatus` và `TransactionDayIndex = floor(TransactionDT/86400)`. DayIndex là chỉ số ngày tương đối, không phải ngày lịch. Các nhãn thiếu là biểu diễn sự chưa biết, **không phải đã khôi phục dữ liệu**.

Code ghi từng lô vào Parquet, đóng writer trước khi kiểm tra. Số dòng in tiến độ không được dùng để tự gán cờ PASS.


In [3]:
def transform_chunks(transaction_path, identity_path, parquet_path, tx_columns, id_columns, chunk_size):
    """Chỉ chuyển đổi dữ liệu; không tự khẳng định quality_checks."""
    identity = pd.read_csv(identity_path, **csv_options(id_columns)).set_index("TransactionID")
    id_fields = [c for c in id_columns if c != "TransactionID"]
    writer = None
    processed = 0
    try:
        for chunk in pd.read_csv(transaction_path, chunksize=chunk_size, **csv_options(tx_columns)):
            present = chunk.TransactionID.isin(identity.index)
            merged = chunk.join(identity, on="TransactionID", how="left", validate="one_to_one")
            derived = pd.DataFrame({"HasIdentity": present.astype(bool),
                "IdentityStatus": pd.Series(np.where(present, "Có bản ghi identity", NO_IDENTITY), index=chunk.index, dtype="string"),
                "TransactionDayIndex": (chunk.TransactionDT // 86400).astype("int64")}, index=chunk.index)
            merged = pd.concat([merged, derived], axis=1)
            for c in merged.columns:
                if categorical(c):
                    values = merged[c].astype("string").str.strip().replace("", pd.NA)
                    if c in id_fields:
                        values = values.mask(~present, NO_IDENTITY).fillna(MISSING_IDENTITY_FIELD)
                    elif c != "IdentityStatus":
                        values = values.fillna(UNKNOWN)
                    merged[c] = values
            merged["isFraud"] = merged["isFraud"].astype("int8")
            table = pa.Table.from_pandas(merged, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression="snappy")
            writer.write_table(table)
            processed += len(chunk)
            if processed % 50000 == 0:
                print(f"Làm sạch và ghi Parquet: {processed:,} dòng", flush=True)
    finally:
        if writer is not None:
            writer.close()
    if processed == 0:
        raise ValueError("File transaction rỗng")
    print(f"Hoàn thành ghi {processed:,} dòng. Chưa kết luận chất lượng trước bước kiểm tra độc lập.")
    del identity
    gc.collect()

transform_chunks(TX_PATH, ID_PATH, PARQUET_PATH, tx_audit["columns"], id_audit["columns"], CHUNK_SIZE)
print("Parquet:", PARQUET_PATH.name, "| bytes:", PARQUET_PATH.stat().st_size)


Làm sạch và ghi Parquet: 50,000 dòng


Làm sạch và ghi Parquet: 100,000 dòng


Làm sạch và ghi Parquet: 150,000 dòng


Làm sạch và ghi Parquet: 200,000 dòng


Làm sạch và ghi Parquet: 250,000 dòng


...

Làm sạch và ghi Parquet: 400,000 dòng


Làm sạch và ghi Parquet: 450,000 dòng


Làm sạch và ghi Parquet: 500,000 dòng


Làm sạch và ghi Parquet: 550,000 dòng


Hoàn thành ghi 590,540 dòng. Chưa kết luận chất lượng trước bước kiểm tra độc lập.
Parquet: fraud_train_clean.parquet | bytes: 160819367


## 4. Đọc lại Parquet và kiểm tra độc lập
Ô code đọc file Parquet đã ghi bằng từng batch, lấy số liệu, schema Arrow và dtype Pandas thực tế. Sau đó đối chiếu với lượt kiểm tra CSV ở Bước 2:

- Số dòng, số ID duy nhất, số fraud; tập ID không mất/thừa.
- Nhãn fraud, số tiền và thời gian theo từng ID; kiểm tra cờ identity bằng sự tồn tại khóa trong CSV identity.
- Missing theo từng cột và nhãn thể hiện nguyên nhân thiếu; số giá trị vô hạn, label không hợp lệ, kiểu dữ liệu không ổn định.
- Trùng ID đầu ra toàn cục. Với trùng toàn dòng Parquet: nếu ID duy nhất thì toàn dòng tất yếu duy nhất; nếu có nhóm ID trùng thì đọc lại và so toàn bộ cột các dòng ứng viên.

Mỗi cờ `passed` là phép so sánh `expected == observed`. Các kiểm tra này chứng minh những thuộc tính đã liệt kê; **không tuyên bố so từng giá trị của mọi đặc trưng V/id**. Missing trước/sau còn được kiểm tra riêng theo từng cột.


In [4]:
def audit_parquet(path, db, chunk_size):
    """Đọc lại từng batch Parquet; số liệu hoàn toàn từ file đầu ra đã đóng writer."""
    db.execute("DROP TABLE IF EXISTS parquet_output")
    db.execute("CREATE TABLE parquet_output(id INTEGER, fraud INTEGER, amount REAL, dt INTEGER, has_identity INTEGER, day_index INTEGER)")
    nulls, labels, no_record, within_missing = Counter(), Counter(), Counter(), Counter()
    min_values, max_values, dtypes = {}, {}, {}
    dtype_changes = invalid_label = invalid_id = infinity = rows = 0
    negative_amount = missing_amount = missing_dt = negative_dt = status_mismatch = 0
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=chunk_size, use_threads=False):
        frame = batch.to_pandas()
        current_types = {c: str(t) for c, t in frame.dtypes.items()}
        if dtypes:
            dtype_changes += int(current_types != dtypes)
        else:
            dtypes = current_types
        nulls.update(frame.isna().sum().to_dict())
        invalid_label += int((~frame.isFraud.isin([0, 1])).sum())
        invalid_id += int(frame.TransactionID.isna().sum())
        negative_amount += int(frame.TransactionAmt.lt(0).sum())
        missing_amount += int(frame.TransactionAmt.isna().sum())
        missing_dt += int(frame.TransactionDT.isna().sum())
        negative_dt += int(frame.TransactionDT.lt(0).sum())
        status_mismatch += int((frame.IdentityStatus != np.where(frame.HasIdentity, "Có bản ghi identity", NO_IDENTITY)).sum())
        numeric = frame.select_dtypes(include="number")
        infinity += int(np.isinf(numeric.to_numpy(dtype="float64")).sum())
        for c, v in numeric.min().items():
            if pd.notna(v):
                min_values[c] = min(min_values.get(c, v), v)
        for c, v in numeric.max().items():
            if pd.notna(v):
                max_values[c] = max(max_values.get(c, v), v)
        for c in frame.columns:
            if categorical(c):
                labels[c] += int(frame[c].isin(MISSING_LABELS).sum())
                no_record[c] += int(frame[c].eq(NO_IDENTITY).sum())
                within_missing[c] += int(frame[c].eq(MISSING_IDENTITY_FIELD).sum())
        records = frame[["TransactionID", "isFraud", "TransactionAmt", "TransactionDT", "HasIdentity", "TransactionDayIndex"]]
        db.executemany("INSERT INTO parquet_output VALUES(?,?,?,?,?,?)", records.itertuples(index=False, name=None))
        rows += len(frame)
        if rows % 100000 == 0:
            print(f"Đọc lại Parquet độc lập: {rows:,} dòng", flush=True)
    db.execute("CREATE INDEX parquet_output_id_idx ON parquet_output(id)")
    db.commit()
    n, unique, fraud, matched = db.execute("SELECT COUNT(*), COUNT(DISTINCT id), SUM(fraud=1), SUM(has_identity) FROM parquet_output").fetchone()
    return dict(rows=n, distinct_ids=unique, frauds=fraud, matched=matched, nulls=dict(nulls), labels=dict(labels),
                no_record=dict(no_record), within_missing=dict(within_missing), pandas_dtypes=dtypes,
                arrow_dtypes={f.name: str(f.type) for f in pf.schema_arrow}, dtype_changes=dtype_changes,
                invalid_label=invalid_label, invalid_id=invalid_id, infinity=infinity,
                negative_amount=negative_amount, missing_amount=missing_amount, missing_dt=missing_dt,
                negative_dt=negative_dt, status_mismatch=status_mismatch, min=min_values, max=max_values,
                metadata_rows=pf.metadata.num_rows, duplicates=duplicate_stats(db, "parquet_output"))

def independent_checks(db, tx, identity, expected, output, original_columns, id_columns):
    checks = []
    def add(name, expected_value, observed_value, explanation):
        checks.append(dict(check=name, expected=expected_value, observed=observed_value,
                           passed=bool(expected_value == observed_value), explanation=explanation))
    add("row_count_preserved", tx["rows"], output["rows"], "csv.reader nguồn vs iter_batches Parquet")
    add("unique_transaction_ids_preserved", tx["distinct_ids"], output["distinct_ids"], "COUNT DISTINCT ở hai bảng kiểm tra riêng")
    add("fraud_count_preserved", tx["frauds"], output["frauds"], "Đếm nhãn bằng 1 độc lập từ nguồn và Parquet")
    for title, clause in [
        ("missing_source_ids", "SELECT COUNT(*) FROM source_transaction t WHERE NOT EXISTS (SELECT 1 FROM parquet_output p WHERE p.id=t.id)"),
        ("unexpected_output_ids", "SELECT COUNT(*) FROM parquet_output p WHERE NOT EXISTS (SELECT 1 FROM source_transaction t WHERE t.id=p.id)"),
        ("fraud_label_mismatches_by_id", "SELECT COUNT(*) FROM source_transaction t JOIN parquet_output p ON t.id=p.id WHERE t.fraud IS NOT p.fraud"),
        ("amount_mismatches_by_id", "SELECT COUNT(*) FROM source_transaction t JOIN parquet_output p ON t.id=p.id WHERE t.amount IS NOT p.amount"),
        ("time_mismatches_by_id", "SELECT COUNT(*) FROM source_transaction t JOIN parquet_output p ON t.id=p.id WHERE t.dt IS NOT p.dt"),
        ("identity_flag_mismatches_by_id", "SELECT COUNT(*) FROM parquet_output p WHERE p.has_identity != EXISTS(SELECT 1 FROM source_identity i WHERE i.id=p.id)"),
        ("relative_day_mismatches", "SELECT COUNT(*) FROM parquet_output p JOIN source_transaction t ON t.id=p.id WHERE p.day_index != CAST(t.dt/86400 AS INTEGER)")]:
        add(title, 0, db.execute(clause).fetchone()[0], "Đối chiếu khóa/giá trị bằng SQL trên dữ liệu đọc riêng")
    add("identity_coverage_preserved", expected["matched"], output["matched"], "Giao khóa nguồn vs cờ HasIdentity đầu ra")
    add("parquet_metadata_rows_match_scan", output["metadata_rows"], output["rows"], "Metadata vs số dòng đọc thực tế")
    add("column_count", len(original_columns) + 3, len(output["arrow_dtypes"]), "Cột nguồn sau join + 3 cột dẫn xuất")
    for tag, audit in [("source_transaction", tx), ("source_identity", identity)]:
        source_fields = ["missing_id", "invalid_id", "reserved_label_collisions"]
        if "isFraud" in audit["columns"]:
            source_fields.append("invalid_label")
        for field in source_fields:
            add(f"{tag}_{field}", 0, audit[field], "Đo trực tiếp trong lượt đọc nguồn")
        add(f"{tag}_duplicate_id_extra_rows", 0, audit["duplicates"]["duplicate_id_extra_rows"], "Trùng khóa toàn file, kể cả khác lô")
    for field in ["dtype_changes", "invalid_label", "invalid_id", "infinity", "negative_amount", "missing_amount", "missing_dt", "negative_dt", "status_mismatch"]:
        add(f"output_{field}", 0, output[field], "Đo khi đọc lại Parquet")
    add("output_duplicate_id_extra_rows", 0, output["duplicates"]["duplicate_id_extra_rows"], "Không trùng khóa sau join")
    null_mismatch = label_mismatch = identity_reason_mismatch = 0
    for c in original_columns:
        wanted_nulls = 0 if categorical(c) else expected["before"].get(c, 0)
        null_mismatch += int(output["nulls"].get(c, 0) != wanted_nulls)
        if categorical(c):
            label_mismatch += int(output["labels"].get(c, 0) != expected["unknown"].get(c, 0))
            if c in id_columns and c != "TransactionID":
                identity_reason_mismatch += int(output["no_record"].get(c, 0) != expected["absent"])
                identity_reason_mismatch += int(output["within_missing"].get(c, 0) != expected["within_normalized"].get(c, 0))
    add("columns_with_wrong_null_counts", 0, null_mismatch, "Theo từng cột: số giữ NaN; categorical chuyển nhãn")
    add("columns_with_wrong_missing_label_counts", 0, label_mismatch, "Nhãn thiếu phải bằng số ô thiếu nguồn sau chuẩn hóa")
    add("identity_missing_reason_count_mismatches", 0, identity_reason_mismatch, "Tách không có record và thiếu trường trong record")
    add("derived_columns_null_count", 0, sum(output["nulls"].get(c, 0) for c in ["HasIdentity", "IdentityStatus", "TransactionDayIndex"]), "Cột dẫn xuất phải có giá trị đầy đủ")
    return pd.DataFrame(checks)

output_audit = audit_parquet(PARQUET_PATH, DB, CHUNK_SIZE)
checks = independent_checks(DB, tx_audit, id_audit, join_expected, output_audit, original_columns, id_audit["columns"])
hash_checks = []
for label, path, source in [("transaction", TX_PATH, tx_audit), ("identity", ID_PATH, id_audit)]:
    actual_hash = file_hash(path)
    hash_checks.append({"check": f"{label}_source_hash_unchanged", "expected": source["sha256"], "observed": actual_hash,
                        "passed": actual_hash == source["sha256"], "explanation": "SHA256 trước và sau xử lý"})
checks = pd.concat([checks, pd.DataFrame(hash_checks)], ignore_index=True)
save_csv(checks, REPORTS / "quality_checks.csv")
display(checks[["check", "expected", "observed", "passed"]])
print(f"Kiểm tra đạt: {int(checks.passed.sum())}/{len(checks)}")


Đọc lại Parquet độc lập: 100,000 dòng


Đọc lại Parquet độc lập: 200,000 dòng


Đọc lại Parquet độc lập: 300,000 dòng


Đọc lại Parquet độc lập: 400,000 dòng


Đọc lại Parquet độc lập: 500,000 dòng


,check,expected,observed,passed
0,row_count_preserved,590540,590540,True
1,unique_transaction_ids_preserved,590540,590540,True
2,fraud_count_preserved,20663,20663,True
3,missing_source_ids,0,0,True
4,unexpected_output_ids,0,0,True
5,fraud_label_mismatches_by_id,0,0,True
6,amount_mismatches_by_id,0,0,True
7,time_mismatches_by_id,0,0,True
8,identity_flag_mismatches_by_id,0,0,True
9,relative_day_mismatches,0,0,True


Kiểm tra đạt: 38/38


## 5. Data dictionary, Stevens, tổng hợp và checklist nghiệm thu
**Data dictionary** có một dòng cho mỗi cột đầu ra: nguồn cột, chính sách đọc CSV (CSV không có dtype nội tại), dtype Pandas sau khi đọc Parquet, kiểu Arrow thực tế, missing, min/max số và đánh giá Stevens.

**Các quyết định cần người review biết:**
- Mã/nhãn được xử lý theo danh nghĩa. Categorical được xử lý bảo thủ như danh nghĩa khi chưa biết thứ tự mức; điều này không khẳng định mọi biến categorical ẩn nghĩa đã được chứng minh là nominal.
- Số tiền: tỷ lệ. Tọa độ thời gian tương đối: khoảng theo cách dùng làm trục thời gian.
- Nhóm C/dist: chỉ đề xuất tỷ lệ có điều kiện; cột `stevens_scale` để trống tới khi xác nhận ý nghĩa gốc 0 và phép biến đổi.
- Nhóm D, V, id_01–id_11: chưa đủ thông tin. Giá trị âm hoặc phân bố rời rạc không đủ kết luận thang đo/chu kỳ.
- Không có biến nào được gán thứ bậc nếu chưa có bằng chứng về thứ tự.

**Mẫu số missing:** so sánh sau join trước làm sạch với sau làm sạch trên cùng các cột gốc (434 cột ở bộ này); không dùng 3 cột dẫn xuất để làm tỷ lệ thiếu có vẻ giảm. Báo cả null và null + nhãn thiếu. Missing bên trong identity dùng mẫu số là số giao dịch có identity, không phải toàn bộ giao dịch.

**Các file giữ lại để review:** 5 bảng CSV (summary, checks, duplicates, missing, dictionary), `run_summary.json` (chỉ số/cờ/hash), `execution.log` và `execution_environment.json` (do công cụ chạy notebook ghi). Các bảng đều hiển thị phần cần đọc ngay dưới đây. CSV chỉ giúp lọc/đối chiếu chi tiết, có thể tạo lại bằng Run All.


In [5]:
def stevens_metadata(name):
    base = dict(stevens_scale="", candidate_scale="", scale_status="Chưa đủ thông tin", confidence="Thấp",
                basis="Tên/giá trị số không đủ chứng minh ý nghĩa phép đo.", limitation="Không diễn giải ý nghĩa nghiệp vụ khi chưa có codebook.",
                source_reference=KAGGLE_URL)
    if name in {"TransactionID", "isFraud", "HasIdentity", "IdentityStatus"}:
        return base | dict(stevens_scale="Danh nghĩa", scale_status="Có căn cứ", confidence="Cao",
                           basis="Mã định danh hoặc nhãn nhóm; 0/1 không làm biến trở thành thang tỷ lệ.", limitation="Dùng đếm/tỷ lệ; không lấy trung bình TransactionID.")
    if categorical(name):
        return base | dict(stevens_scale="Danh nghĩa", scale_status="Quy ước phân tích bảo thủ", confidence="Trung bình",
                           basis="Nguồn liệt kê categorical; chưa có thứ tự mức được xác nhận nên xử lý như danh nghĩa.",
                           limitation="Categorical không tự chứng minh nominal; cần codebook để xác nhận/bổ sung thứ tự cho từng biến ẩn nghĩa.")
    if name == "TransactionAmt":
        return base | dict(stevens_scale="Tỷ lệ", scale_status="Có căn cứ", confidence="Cao", basis="Số tiền thanh toán: gốc 0 mang nghĩa không có số tiền.",
                           limitation="So sánh tỷ lệ trong cùng đơn vị; tổng tiền giao dịch fraud không phải thiệt hại đã xác nhận.", source_reference=DETAIL_URL)
    if name in {"TransactionDT", "TransactionDayIndex"}:
        return base | dict(stevens_scale="Khoảng", scale_status="Có căn cứ theo cách sử dụng", confidence="Trung bình",
                           basis="Vị trí trên trục thời gian tính từ mốc tùy ý; DayIndex=floor(TransactionDT/86400).",
                           limitation="Không suy ra ngày lịch/giờ địa phương; tỷ lệ hai tọa độ thời gian không có ý nghĩa. Chênh lệch thời lượng là khái niệm khác.")
    if re.fullmatch(r"C(?:[1-9]|1[0-4])", name) or name in {"dist1", "dist2"}:
        return base | dict(candidate_scale="Tỷ lệ", scale_status="Đề xuất có điều kiện", confidence="Trung bình",
                           basis="Mô tả nhóm là đếm/khoảng cách; chưa xác nhận biến cụ thể là đại lượng gốc chưa biến đổi.",
                           limitation="Chỉ chốt tỷ lệ khi xác nhận ý nghĩa gốc 0, đơn vị và phép biến đổi; không tự gán tên nghiệp vụ.", source_reference=DETAIL_URL)
    if re.fullmatch(r"D(?:[1-9]|1[0-5])", name):
        return base | dict(candidate_scale="Khoảng hoặc tỷ lệ", basis="Nhóm timedelta; thiếu mốc và phép biến đổi cụ thể.",
                           limitation="Giá trị âm không tự phủ định thang tỷ lệ. D9 có phân bố rời rạc cũng không đủ chứng minh chu kỳ.", source_reference=DETAIL_URL)
    return base | dict(basis="V là đặc trưng được tạo; id_01–id_11 ẩn nghĩa. Kiểu float không xác định thang Stevens.")

def build_reports(tx, identity, expected, output, checks, report_dir):
    originals = tx["columns"] + [c for c in identity["columns"] if c != "TransactionID"]
    n = output["rows"]
    id_fields = set(identity["columns"]) - {"TransactionID"}
    missing_rows, dictionary = [], []
    for c in output["arrow_dtypes"]:
        source = "transaction" if c in tx["columns"] else "identity" if c in id_fields else "derived"
        before = expected["before"].get(c, 0)
        normalized = expected["unknown"].get(c, 0)
        is_original = c in originals
        missing_rows.append(dict(column=c, source=source, comparison_scope="original" if is_original else "derived",
            denominator_rows=n, missing_after_join_before_clean=before, missing_before_pct=100*before/n,
            missing_after_normalizing_blanks=normalized, output_null_count=output["nulls"].get(c, 0),
            output_null_pct=100*output["nulls"].get(c, 0)/n,
            structural_missing_no_identity=expected["absent"] if c in id_fields else 0,
            missing_within_matched_identity=expected["within_normalized"].get(c, 0) if c in id_fields else None,
            within_identity_denominator=expected["matched"] if c in id_fields else None,
            missing_within_identity_pct=100*expected["within_normalized"].get(c, 0)/expected["matched"] if c in id_fields and expected["matched"] else None,
            explicit_missing_labels=output["labels"].get(c, 0) if is_original else 0,
            unknown_after_null_plus_label=output["nulls"].get(c, 0)+(output["labels"].get(c, 0) if is_original else 0),
            note="Nhãn thay thế không khôi phục thông tin; không tính cột derived vào tỷ lệ missing so sánh toàn bảng."))
        dictionary.append(dict(column=c, source=source, dtype_csv_read_policy=read_policy([c])[c] if is_original else "derived",
            dtype_after_pandas=output["pandas_dtypes"][c], dtype_after_parquet=output["arrow_dtypes"][c],
            null_count_after=output["nulls"].get(c, 0), observed_min=output["min"].get(c), observed_max=output["max"].get(c),
            missing_treatment="Nhãn phân biệt nguyên nhân" if categorical(c) else "Giữ null; không điền số giả" if is_original else "Tính từ nguồn",
            **stevens_metadata(c)))
    missing = pd.DataFrame(missing_rows)
    dictionary = pd.DataFrame(dictionary)
    def stat(stage, rows, columns, unique, fraud, matched, null_count, unknown_count):
        return dict(stage=stage, rows=rows, columns=columns, unique_transaction_ids=unique, fraud_count=fraud,
                    fraud_pct=100*fraud/rows if fraud is not None else None,
                    matched_identity_rows=matched, identity_coverage_pct=100*matched/rows if matched is not None else None,
                    missing_cells=null_count, missing_denominator_cells=rows*columns, missing_pct=100*null_count/(rows*columns),
                    unknown_cells_after_normalization=unknown_count)
    final_null = sum(output["nulls"].get(c, 0) for c in originals)
    final_unknown = final_null + sum(output["labels"].get(c, 0) for c in originals)
    summary = pd.DataFrame([
        stat("Nguồn transaction", tx["rows"], len(tx["columns"]), tx["distinct_ids"], tx["frauds"], expected["matched"], sum(tx["empty"].values()), sum(tx["normalized_missing"].values())),
        stat("Nguồn identity (không có nhãn fraud)", identity["rows"], len(identity["columns"]), identity["distinct_ids"], None, None, sum(identity["empty"].values()), sum(identity["normalized_missing"].values())),
        stat("Sau left join, trước làm sạch (cột gốc)", tx["rows"], len(originals), tx["distinct_ids"], tx["frauds"], expected["matched"], sum(expected["before"].values()), sum(expected["unknown"].values())),
        stat("Parquet sau làm sạch (chỉ cột gốc để so sánh)", n, len(originals), output["distinct_ids"], output["frauds"], output["matched"], final_null, final_unknown),
        stat("Parquet đầy đủ (gồm 3 cột dẫn xuất)", n, len(output["arrow_dtypes"]), output["distinct_ids"], output["frauds"], output["matched"], sum(output["nulls"].values()), final_unknown)])
    duplicates = pd.DataFrame([tx["duplicates"], identity["duplicates"], output["duplicates"]])
    for name, frame in [("summary_before_after", summary), ("quality_checks", checks), ("duplicates", duplicates),
                        ("missing_before_after", missing), ("data_dictionary", dictionary)]:
        save_csv(frame, report_dir / f"{name}.csv")
    return summary, duplicates, missing, dictionary

summary, duplicates, missing, dictionary = build_reports(tx_audit, id_audit, join_expected, output_audit, checks, REPORTS)
print("BẢNG TRƯỚC / SAU")
display(summary)
print("TRÙNG KHÓA VÀ TRÙNG TOÀN DÒNG — KHÔNG XÓA DÒNG")
display(duplicates.drop(columns=["full_row_definition", "action"]))
print("MISSING: 12 CỘT CÓ TỶ LỆ THIẾU TRƯỚC LÀM SẠCH CAO NHẤT")
display(missing.sort_values("missing_before_pct", ascending=False).head(12))
print("KIỂU DỮ LIỆU THỰC TẾ — TOÀN BỘ CỘT")
display(dictionary.groupby(["dtype_after_pandas", "dtype_after_parquet"], dropna=False).size().rename("columns").reset_index())
print("TỔNG HỢP MỨC ĐỘ KẾT LUẬN STEVENS")
display(dictionary.groupby(["scale_status", "stevens_scale", "candidate_scale"], dropna=False).size().rename("columns").reset_index())
print("VÍ DỤ DICTIONARY; BẢN ĐẦY ĐỦ CÓ", len(dictionary), "DÒNG TRONG data_dictionary.csv")
display(dictionary[dictionary.column.isin(["TransactionID", "isFraud", "TransactionAmt", "TransactionDT", "card1", "C1", "D9", "V1", "id_01", "HasIdentity"])])
uncertain = dictionary[dictionary.scale_status.isin(["Chưa đủ thông tin", "Đề xuất có điều kiện"])]
print("Danh sách biến chưa chốt thang Stevens (bao gồm đề xuất có điều kiện):")
print(", ".join(uncertain.column))

run_summary = {
    "started_utc": STARTED_AT, "finished_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds_to_report": round(time.perf_counter() - START_TIMER, 3), "chunk_size": CHUNK_SIZE,
    "input_rows": {"transaction": tx_audit["rows"], "identity": id_audit["rows"]},
    "output_rows": output_audit["rows"], "output_columns": len(dictionary),
    "fraud_rows": output_audit["frauds"], "fraud_pct": 100*output_audit["frauds"]/output_audit["rows"],
    "matched_identity_rows": output_audit["matched"], "orphan_identity_rows": join_expected["orphan_identity"],
    "source_sha256": {TX_PATH.name: tx_audit["sha256"], ID_PATH.name: id_audit["sha256"]},
    "parquet_sha256": file_hash(PARQUET_PATH),
    "quality_checks": {str(r.check): bool(r.passed) for r in checks.itertuples()},
    "all_checks_passed": bool(checks.passed.all()),
    "scope_limit": "Đối chiếu ID/label/amount/time/identity và missing từng cột; không so từng giá trị của mọi đặc trưng ẩn nghĩa.",
    "stevens_unresolved_including_conditional": len(uncertain),
    "numeric_policy": "float64; giữ null; categorical dùng nhãn nguồn gốc missing",
    "versions": {name: importlib.metadata.version(name) for name in ["pandas", "numpy", "pyarrow", "ipykernel"]},
}
(REPORTS / "run_summary.json").write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
DB.close()
# Chỉ xóa file SQLite tạm do notebook này tạo; không xóa dữ liệu nguồn/kết quả.
(REPORTS / "audit_work.sqlite").unlink(missing_ok=True)
print("CHECKLIST:")
print("1. Các ô code đã chạy tuần tự; log và bản notebook có output được lưu bởi công cụ thực thi.")
print("2. Kiểm tra nguồn/Parquet:", int(checks.passed.sum()), "/", len(checks), "đạt.")
print("3. Dictionary:", len(dictionary), "cột có dtype thực tế.")
print("4. Duplicate: đã báo cáo riêng ID và toàn dòng, không tự xóa.")
print("5. Đã xuất summary_before_after.csv và missing_before_after.csv.")
print("6. Stevens:", len(uncertain), "biến chưa chốt hoặc có điều kiện; không ép kết luận.")
if not checks.passed.all():
    raise AssertionError("Có kiểm tra không đạt. Đã lưu bằng chứng để xem xét; chưa đủ điều kiện nghiệm thu.")
print("KẾT LUẬN KỸ THUẬT: Các kiểm tra trong phạm vi đã đạt. Nhóm trưởng đối chiếu checklist để chốt Giai đoạn 1.")


BẢNG TRƯỚC / SAU


,stage,rows,columns,unique_transaction_ids,fraud_count,fraud_pct,matched_identity_rows,identity_coverage_pct,missing_cells,missing_denominator_cells,missing_pct,unknown_cells_after_normalization
0,Nguồn transaction,590540,394,590540,20663.0,3.499001,144233.0,24.423917,95566686,232672760,41.073431,95566686
1,Nguồn identity (không có nhãn fraud),144233,41,144233,NaN,NaN,NaN,NaN,2104107,5913553,35.581097,2104107
2,"Sau left join, trước làm sạch (cột gốc)",590540,434,590540,20663.0,3.499001,144233.0,24.423917,115523073,256294360,45.074372,115523073
3,Parquet sau làm sạch (chỉ cột gốc để so sánh),590540,434,590540,20663.0,3.499001,144233.0,24.423917,97719626,256294360,38.127888,115523073
4,Parquet đầy đủ (gồm 3 cột dẫn xuất),590540,437,590540,20663.0,3.499001,144233.0,24.423917,97719626,258065980,37.866140,115523073


TRÙNG KHÓA VÀ TRÙNG TOÀN DÒNG — KHÔNG XÓA DÒNG


,dataset,duplicate_id_groups,duplicate_id_rows_involved,duplicate_id_extra_rows,duplicate_full_row_groups,duplicate_full_rows_involved,duplicate_full_extra_rows
0,source_transaction,0,0,0,0,0,0
1,source_identity,0,0,0,0,0,0
2,parquet_output,0,0,0,0,0,0


MISSING: 12 CỘT CÓ TỶ LỆ THIẾU TRƯỚC LÀM SẠCH CAO NHẤT


...

,scale_status,stevens_scale,candidate_scale,columns
0,Chưa đủ thông tin,,,350
1,Chưa đủ thông tin,,Khoảng hoặc tỷ lệ,15
2,Có căn cứ,Danh nghĩa,,4
3,Có căn cứ,Tỷ lệ,,1
4,Có căn cứ theo cách sử dụng,Khoảng,,2
5,Quy ước phân tích bảo thủ,Danh nghĩa,,49
6,Đề xuất có điều kiện,,Tỷ lệ,16


VÍ DỤ DICTIONARY; BẢN ĐẦY ĐỦ CÓ 437 DÒNG TRONG data_dictionary.csv


,column,source,dtype_csv_read_policy,dtype_after_pandas,dtype_after_parquet,null_count_after,observed_min,observed_max,missing_treatment,stevens_scale,candidate_scale,scale_status,confidence,basis,limitation,source_reference
0,TransactionID,transaction,int64,int64,int64,0,2987000.000,3.577539e+06,Giữ null; không điền số giả,Danh nghĩa,,Có căn cứ,Cao,Mã định danh hoặc nhãn nhóm; 0/1 không làm biế...,Dùng đếm/tỷ lệ; không lấy trung bình Transacti...,https://www.kaggle.com/c/ieee-fraud-detection/...
1,isFraud,transaction,int64,int8,int8,0,0.000,1.000000e+00,Giữ null; không điền số giả,Danh nghĩa,,Có căn cứ,Cao,Mã định danh hoặc nhãn nhóm; 0/1 không làm biế...,Dùng đếm/tỷ lệ; không lấy trung bình Transacti...,https://www.kaggle.com/c/ieee-fraud-detection/...
2,TransactionDT,transaction,int64,int64,int64,0,86400.000,1.581113e+07,Giữ null; không điền số giả,Khoảng,,Có căn cứ theo cách sử dụng,Trung bình,Vị trí trên trục thời gian tính từ mốc tùy ý; ...,Không suy ra ngày lịch/giờ địa phương; tỷ lệ h...,https://www.kaggle.com/c/ieee-fraud-detection/...
3,TransactionAmt,transaction,float64,float64,double,0,0.251,3.193739e+04,Giữ null; không điền số giả,Tỷ lệ,,Có căn cứ,Cao,Số tiền thanh toán: gốc 0 mang nghĩa không có ...,So sánh tỷ lệ trong cùng đơn vị; tổng tiền gia...,https://www.kaggle.com/competitions/ieee-fraud...
5,card1,transaction,string,string,large_string,0,NaN,NaN,Nhãn phân biệt nguyên nhân,Danh nghĩa,,Quy ước phân tích bảo thủ,Trung bình,Nguồn liệt kê categorical; chưa có thứ tự mức ...,Categorical không tự chứng minh nominal; cần c...,https://www.kaggle.com/c/ieee-fraud-detection/...
17,C1,transaction,float64,float64,double,0,0.000,4.685000e+03,Giữ null; không điền số giả,,Tỷ lệ,Đề xuất có điều kiện,Trung bình,Mô tả nhóm là đếm/khoảng cách; chưa xác nhận b...,"Chỉ chốt tỷ lệ khi xác nhận ý nghĩa gốc 0, đơn...",https://www.kaggle.com/competitions/ieee-fraud...
39,D9,transaction,float64,float64,double,515614,0.000,9.583330e-01,Giữ null; không điền số giả,,Khoảng hoặc tỷ lệ,Chưa đủ thông tin,Thấp,Nhóm timedelta; thiếu mốc và phép biến đổi cụ ...,Giá trị âm không tự phủ định thang tỷ lệ. D9 c...,https://www.kaggle.com/competitions/ieee-fraud...
55,V1,transaction,float64,float64,double,279287,0.000,1.000000e+00,Giữ null; không điền số giả,,,Chưa đủ thông tin,Thấp,V là đặc trưng được tạo; id_01–id_11 ẩn nghĩa....,Không diễn giải ý nghĩa nghiệp vụ khi chưa có ...,https://www.kaggle.com/c/ieee-fraud-detection/...
394,id_01,identity,float64,float64,double,446307,-100.000,0.000000e+00,Giữ null; không điền số giả,,,Chưa đủ thông tin,Thấp,V là đặc trưng được tạo; id_01–id_11 ẩn nghĩa....,Không diễn giải ý nghĩa nghiệp vụ khi chưa có ...,https://www.kaggle.com/c/ieee-fraud-detection/...
434,HasIdentity,derived,derived,bool,bool,0,NaN,NaN,Tính từ nguồn,Danh nghĩa,,Có căn cứ,Cao,Mã định danh hoặc nhãn nhóm; 0/1 không làm biế...,Dùng đếm/tỷ lệ; không lấy trung bình Transacti...,https://www.kaggle.com/c/ieee-fraud-detection/...


Danh sách biến chưa chốt thang Stevens (bao gồm đề xuất có điều kiện):
dist1, dist2, C1, C2, C3, C4, C5, C6, C7, C8, C9, C10, C11, C12, C13, C14, D1, D2, D3, D4, D5, D6, D7, D8, D9, D10, D11, D12, D13, D14, D15, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V15, V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, V29, V30, V31, V32, V33, V34, V35, V36, V37, V38, V39, V40, V41, V42, V43, V44, V45, V46, V47, V48, V49, V50, V51, V52, V53, V54, V55, V56, V57, V58, V59, V60, V61, V62, V63, V64, V65, V66, V67, V68, V69, V70, V71, V72, V73, V74, V75, V76, V77, V78, V79, V80, V81, V82, V83, V84, V85, V86, V87, V88, V89, V90, V91, V92, V93, V94, V95, V96, V97, V98, V99, V100, V101, V102, V103, V104, V105, V106, V107, V108, V109, V110, V111, V112, V113, V114, V115, V116, V117, V118, V119, V120, V121, V122, V123, V124, V125, V126, V127, V128, V129, V130, V131, V132, V133, V134, V135, V136, V137, V138, V139, V140, V141, V142, V143, V144, V145, V146, V147, V148, V149, V1

CHECKLIST:
1. Các ô code đã chạy tuần tự; log và bản notebook có output được lưu bởi công cụ thực thi.
2. Kiểm tra nguồn/Parquet: 38 / 38 đạt.
3. Dictionary: 437 cột có dtype thực tế.
4. Duplicate: đã báo cáo riêng ID và toàn dòng, không tự xóa.
5. Đã xuất summary_before_after.csv và missing_before_after.csv.
6. Stevens: 381 biến chưa chốt hoặc có điều kiện; không ép kết luận.
KẾT LUẬN KỸ THUẬT: Các kiểm tra trong phạm vi đã đạt. Nhóm trưởng đối chiếu checklist để chốt Giai đoạn 1.
